# SSL vs Baseline – Comparație completă (Kaggle)
**Proiect:** Language-specific correction for Romanian

## Înainte de rulare:
1. Settings (dreapta) → **Accelerator → GPU P100** ✅
2. Settings → **Add data** → selectează dataset-ul `NLP_Project` ✅
3. Rulează celulele în ordine

Progress bar și erori vizibile live în fiecare celulă.

In [ ]:
# ── Celula 1: Setup pip ────────────────────────────────────────────────────
import subprocess, sys

# ── Test rapid CUDA înainte de orice ──────────────────────────────────────
import torch as _t
if _t.cuda.is_available():
    cap = _t.cuda.get_device_capability(0)
    print(f'GPU: {_t.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}')
    cuda_url = 'https://download.pytorch.org/whl/cu118' if cap[0] < 7 else 'https://download.pytorch.org/whl/cu121'
else:
    cuda_url = 'https://download.pytorch.org/whl/cu121'
    print('CUDA nu detectat, folosesc cu121')

print(f'Reinstalare torch neconditionata ({cuda_url.split("/")[-1]})...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.3.1', 'torchvision',
    '--index-url', cuda_url], capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])
    raise RuntimeError('pip install esuat')
print('torch reinstalat OK')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'scikit-learn', 'pandas', 'tqdm', 'matplotlib'], check=True)

print()
print('=' * 55)
print('RESTART SESSION ACUM pentru a activa torch nou!')
print('  Kaggle: butonul cu sageata sus -> Restart Session')
print('  Apoi ruleaza celulele 2-8 in ordine')
print('=' * 55)

In [ ]:
# ── Celula 2: Setup directoare + detectare dataset ────────────────────────
import shutil, os, sys
from pathlib import Path

os.chdir('/kaggle/working')
for d in ['src', 'data/prepared', 'results/detector_baseline',
          'results/detector_ssl', 'results/ssl_dae']:
    Path(d).mkdir(parents=True, exist_ok=True)

# Detectează automat dataset-ul NLP_Project
input_root = Path('/kaggle/input')
print('Datasets disponibile:')
for d in sorted(input_root.iterdir()):
    print(f'  {d}')

DATASET = None
for d in input_root.iterdir():
    if 'nlp' in d.name.lower() or 'project' in d.name.lower():
        DATASET = d; break
if DATASET is None:
    DATASET = sorted(input_root.iterdir())[0]

print(f'\n✅ Dataset: {DATASET}')
sys.path.insert(0, '/kaggle/working/src')

In [ ]:
# ── Celula 3: Copiere fișiere în working dir ───────────────────────────────
import shutil
from pathlib import Path

DATASET = '/kaggle/input/datasets/alexandruduca/nlpssl-project'

# Copiază fișierele src/
src_files = ['utils.py', 'detector.py', 'data_prep.py',
             'ssl_trainer.py', 'ssl_corruption.py', 'prepare_unlabeled_corpus.py']

for fname in src_files:
    # Caută fișierul recursiv în dataset
    matches = list(Path(DATASET).rglob(fname))
    if matches:
        shutil.copy(str(matches[0]), f'/kaggle/working/src/{fname}')
        print(f'✅ src/{fname}')
    else:
        print(f'❌ LIPSĂ: {fname}')

# Copiază full_dataset.csv
csv_matches = list(Path(DATASET).rglob('full_dataset.csv'))
if csv_matches:
    shutil.copy(str(csv_matches[0]), '/kaggle/working/data/full_dataset.csv')
    size = Path('/kaggle/working/data/full_dataset.csv').stat().st_size / 1024**2
    print(f'✅ data/full_dataset.csv ({size:.1f} MB)')
else:
    print('❌ full_dataset.csv LIPSĂ din dataset!')

In [ ]:
# ── Celula 4: Helper streaming + Pregătire date ───────────────────────────
import subprocess, sys, json
from pathlib import Path

def run_live(cmd):
    """Rulează comandă și afișează output + erori în timp real."""
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Comandă eșuată cu codul {proc.returncode}')

print('=== Pregătire date ===')
run_live([
    sys.executable, 'src/data_prep.py',
    '--csv', 'data/full_dataset.csv',
    '--out_dir', 'data/prepared',
    '--val_size', '0.05',
    '--test_size', '0.05',
    '--seed', '42',
])

for split in ['detector_train', 'detector_val', 'detector_test']:
    f = Path(f'data/prepared/{split}.jsonl')
    if f.exists():
        n = sum(1 for _ in open(f))
        print(f'{split}: {n:,} exemple')

In [ ]:
# ── Celula 5: Corpus SSL ──────────────────────────────────────────────────
print('=== Corpus SSL ===')
run_live([
    sys.executable, 'src/prepare_unlabeled_corpus.py',
    '--input', 'data/full_dataset.csv',
    '--output', 'data/unlabeled_corpus_full.txt',
    '--columns', 'correct',
])

n = sum(1 for _ in open('data/unlabeled_corpus_full.txt', encoding='utf-8'))
print(f'✅ Corpus SSL: {n:,} propoziții')

In [ ]:
# ── Celula 6: SSL Pre-training (direct, cu progress bar) ──────────────────
import sys, json, gc, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
from pathlib import Path

sys.path.insert(0, 'src')
from ssl_corruption import TextCorruptor, edit_distance
from utils import set_seed
set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
print(f'Device: {device} | GPUs disponibile: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

MODEL_NAME   = 'readerbench/RoBERT-large'
UNLABELED    = 'data/unlabeled_corpus_full.txt'
OUT_DIR      = Path('results/ssl_dae')
EPOCHS       = 3
# Cu 2x T4: batch_size x n_gpus = batch efectiv per step
BATCH_SIZE   = 16 * max(1, n_gpus)   # 32 cu 2 GPU-uri
MAX_LENGTH   = 96
LR           = 2e-5 * max(1, n_gpus) # LR scaling liniar
GRAD_ACCUM   = 4
MAX_EXAMPLES = 20000

OUT_DIR.mkdir(parents=True, exist_ok=True)

class SSLDataset(Dataset):
    def __init__(self, path, tokenizer, max_length, limit, intensity, ctype):
        self.texts = []
        with open(path, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line: self.texts.append(line)
                if limit > 0 and len(self.texts) >= limit: break
        self.tok = tokenizer; self.max_length = max_length
        self.corruptor = TextCorruptor(); self.intensity = intensity; self.ctype = ctype

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        clean = self.texts[idx]
        for _ in range(5):
            corrupted = self.corruptor.apply_corruption(clean, self.ctype, self.intensity)
            if edit_distance(clean, corrupted) >= 2: break
        enc_c = self.tok(clean,     max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')
        enc_x = self.tok(corrupted, max_length=self.max_length, truncation=True, padding='max_length', return_tensors='pt')
        return {'input_ids': enc_x['input_ids'].squeeze(0),
                'attention_mask': enc_x['attention_mask'].squeeze(0),
                'labels': enc_c['input_ids'].squeeze(0)}

print('Loading model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder   = AutoModel.from_pretrained(MODEL_NAME).to(device)
head = nn.Linear(encoder.config.hidden_size, tokenizer.vocab_size).to(device)

# DataParallel pentru 2x T4
if n_gpus > 1:
    print(f'Activez DataParallel pe {n_gpus} GPU-uri')
    encoder = nn.DataParallel(encoder)
    head    = nn.DataParallel(head)

vocab_size = tokenizer.vocab_size
enc_params  = encoder.module.parameters() if n_gpus > 1 else encoder.parameters()
head_params = head.module.parameters()    if n_gpus > 1 else head.parameters()
optimizer = torch.optim.AdamW(list(enc_params) + list(head_params), lr=LR)
loss_fct  = nn.CrossEntropyLoss(reduction='none')

curriculum = [('light', 0.3), ('medium', 0.55), ('mixed', 0.8)]
best_loss = float('inf')

for epoch in range(EPOCHS):
    ctype, intensity = curriculum[epoch]
    print(f'\n📈 Epoch {epoch+1}/{EPOCHS} | type={ctype} intensity={intensity}')
    ds = SSLDataset(UNLABELED, tokenizer, MAX_LENGTH, MAX_EXAMPLES, intensity, ctype)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    encoder.train(); head.train()
    epoch_loss = 0.0; optimizer.zero_grad()
    pbar = tqdm(dl, desc=f'Epoch {epoch+1}')
    for i, batch in enumerate(pbar):
        batch = {k: v.to(device) for k, v in batch.items()}
        hidden = encoder(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state
        logits = head(hidden)
        loss = loss_fct(logits.view(-1, vocab_size), batch['labels'].view(-1))
        loss = (loss * batch['attention_mask'].view(-1)).sum() / (batch['attention_mask'].sum() + 1e-8)
        (loss / GRAD_ACCUM).backward()
        epoch_loss += loss.item()
        if (i+1) % GRAD_ACCUM == 0:
            all_params = list((encoder.module if n_gpus>1 else encoder).parameters()) + \
                         list((head.module    if n_gpus>1 else head).parameters())
            torch.nn.utils.clip_grad_norm_(all_params, 1.0)
            optimizer.step(); optimizer.zero_grad()
        if i % 20 == 0: pbar.set_postfix({'loss': f'{epoch_loss/(i+1):.4f}'})
    avg = epoch_loss / len(dl)
    print(f'✓ Epoch {epoch+1} avg loss: {avg:.4f}')
    if avg < best_loss:
        best_loss = avg
        best_dir = OUT_DIR / 'best'; best_dir.mkdir(exist_ok=True)
        _enc = encoder.module if n_gpus > 1 else encoder
        _enc.save_pretrained(best_dir); tokenizer.save_pretrained(best_dir)
        print(f'🏆 Best saved (loss={best_loss:.4f})')

(OUT_DIR / 'final').mkdir(exist_ok=True)
_enc = encoder.module if n_gpus > 1 else encoder
_enc.save_pretrained(OUT_DIR / 'final'); tokenizer.save_pretrained(OUT_DIR / 'final')
(OUT_DIR / 'training_info.json').write_text(json.dumps({'best_loss': best_loss, 'epochs': EPOCHS, 'examples': MAX_EXAMPLES}))
print(f'\n✅ SSL complet! Best loss: {best_loss:.4f}')

# Eliberează memorie
del encoder, head, optimizer, ds, dl
gc.collect(); torch.cuda.empty_cache()
print('✅ Memorie GPU eliberată')

In [ ]:
# ── Celula 7: Detector BASELINE ───────────────────────────────────────────
import time
from pathlib import Path

print('🚀 Detector BASELINE (RoBERT-large stock)...')
start = time.time()

run_live([
    sys.executable, 'src/detector.py',
    '--data_dir', 'data/prepared',
    '--out_dir', 'results/detector_baseline',
    '--model_name', 'readerbench/RoBERT-large',
    '--epochs', '3',
    '--batch_size', '16',
    '--max_length', '128',
    '--lr', '2e-5',
    '--grad_accum', '2',
    '--max_train_examples', '20000',
    '--num_workers', '2',
])

print(f'\n⏱ {(time.time()-start)/60:.1f} min')
hist = json.loads(Path('results/detector_baseline/history.json').read_text())
best = max(hist, key=lambda x: x['f05'])
print(f'\n📊 BASELINE Best F0.5: {best["f05"]:.4f} (Epoch {best["epoch"]})')
print(f'   Precision: {best["precision"]:.4f}  Recall: {best["recall"]:.4f}')

In [ ]:
# ── Celula 8: Detector CU SSL encoder ────────────────────────────────────
print('🚀 Detector CU SSL encoder...')
start = time.time()

run_live([
    sys.executable, 'src/detector.py',
    '--data_dir', 'data/prepared',
    '--out_dir', 'results/detector_ssl',
    '--model_name', 'results/ssl_dae/best',
    '--epochs', '3',
    '--batch_size', '16',
    '--max_length', '128',
    '--lr', '2e-5',
    '--grad_accum', '2',
    '--max_train_examples', '20000',
    '--num_workers', '2',
])

print(f'\n⏱ {(time.time()-start)/60:.1f} min')
hist_ssl = json.loads(Path('results/detector_ssl/history.json').read_text())
best_s = max(hist_ssl, key=lambda x: x['f05'])
print(f'\n📊 SSL Best F0.5: {best_s["f05"]:.4f} (Epoch {best_s["epoch"]})')
print(f'   Precision: {best_s["precision"]:.4f}  Recall: {best_s["recall"]:.4f}')

In [ ]:
# ── Celula 9: Comparație + Grafice ────────────────────────────────────────
import json, matplotlib.pyplot as plt, numpy as np
from pathlib import Path

hist_b = json.loads(Path('results/detector_baseline/history.json').read_text())
hist_s = json.loads(Path('results/detector_ssl/history.json').read_text())
best_b = max(hist_b, key=lambda x: x['f05'])
best_s = max(hist_s, key=lambda x: x['f05'])

# Tabel
metrics = [('F0.5', 'f05'), ('Precizie', 'precision'), ('Recall', 'recall'), ('Type Acc', 'type_acc')]
print('='*58)
print(f'{"Metrică":<18} {"Baseline":>10} {"SSL":>10} {"Δ":>10}')
print('='*58)
for lbl, key in metrics:
    bv, sv = best_b.get(key, 0), best_s.get(key, 0)
    d = sv - bv
    print(f'{lbl:<18} {bv:>10.4f} {sv:>10.4f} {"▲" if d>0 else "▼"}{abs(d):>8.4f}')
print('='*58)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 6))
labels = [m[0] for m in metrics]; keys = [m[1] for m in metrics]
x = np.arange(len(labels)); w = 0.35
b_vals = [best_b.get(k,0) for k in keys]
s_vals = [best_s.get(k,0) for k in keys]
bars_b = ax.bar(x - w/2, b_vals, w, label='Baseline', color='#2196F3', alpha=0.85)
bars_s = ax.bar(x + w/2, s_vals, w, label='SSL Pre-training', color='#4CAF50', alpha=0.85)
ax.bar_label(bars_b, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(bars_s, fmt='%.4f', padding=3, fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0.5, 1.05); ax.legend(fontsize=11)
ax.set_title('Baseline vs SSL Pre-training – Detector Erori Gramaticale (RO)', fontsize=13)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('results/comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Grafic salvat: results/comparison_bar.png')